# FRE 521D Final Project
## Drought Impacts on Cereal Production: A Multi-Regional Analysis

**Team:** AgroAnalytics  


**Date:** February 10, 2024

---

## 1. Setup and Imports

Loading all the libraries we need

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')  # suppress annoying warnings

# plot settings - took a while to figure out the right style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('All libraries imported!')

In [ ]:
# database connection stuff
# NOTE: change password before running!!
import sqlite3

# we're using sqlite for this project since its easier to share
# the data is exported from our postgres database

DB_PATH = '../data/fre521d_project.db'

## 2. Data Loading

Loading data from our A1 and A2 databases

In [ ]:
# Actually we're gonna load from CSV since thats what we exported
# the sql queries are in the sql_queries.sql file

df_crops = pd.read_csv('../data/crop_production_cleaned.csv')
df_weather = pd.read_csv('../data/weather_data_cleaned.csv')
df_countries = pd.read_csv('../data/countries.csv')

print(f"Crops: {len(df_crops)} rows")
print(f"Weather: {len(df_weather)} rows")
print(f"Countries: {len(df_countries)} rows")

In [ ]:
# check the data
df_crops.head()

In [ ]:
print(df_crops.columns.tolist())

In [ ]:
df_weather.head()

In [ ]:
# what countries and crops do we have?
print("Countries:", df_crops['country_name'].nunique())
print("Crops:", df_crops['crop'].unique())

## 3. Data Cleaning and Preprocessing

In [ ]:
# check for missing values
print("Missing values in crop data:")
print(df_crops.isnull().sum())
print()
print("Missing values in weather data:")
print(df_weather.isnull().sum())

In [ ]:
# Calculate missing percentage
# Aisha wrote this function
def calc_missing_pct(df):
    missing = df.isnull().sum()
    pct = missing / len(df) * 100
    result = pd.DataFrame({'missing': missing, 'pct': pct})
    return result[result['missing'] > 0].sort_values('pct', ascending=False)

print("Crops missing:")
print(calc_missing_pct(df_crops))

In [ ]:
# remove rows with missing yield - can't analyze without that
df_crops_clean = df_crops.dropna(subset=['yield_kg_ha']).copy()
print(f"Rows before: {len(df_crops)}, after: {len(df_crops_clean)}")

In [ ]:
# merge crop and weather data
# matching on country and year

df_merged = df_crops_clean.merge(
    df_weather,
    on=['iso3_code', 'year'],
    how='inner'
)

print(f"Merged dataset: {len(df_merged)} rows")

In [ ]:
# add country info
df_merged = df_merged.merge(
    df_countries[['iso3_code', 'income_group', 'region']],
    on='iso3_code',
    how='left'
)

print(f"After adding country info: {len(df_merged)} rows")
df_merged.head()

In [ ]:
# Create drought indicator
# we're defining drought as precipitation below 25th percentile for that country

# calculate country-specific precipitation thresholds
precip_thresholds = df_merged.groupby('iso3_code')['precipitation_mm'].quantile(0.25).reset_index()
precip_thresholds.columns = ['iso3_code', 'precip_25pct']

df_merged = df_merged.merge(precip_thresholds, on='iso3_code')

# drought years are when precip is below 25th percentile
df_merged['drought'] = (df_merged['precipitation_mm'] < df_merged['precip_25pct']).astype(int)

print(f"Drought years: {df_merged['drought'].sum()} ({df_merged['drought'].mean()*100:.1f}%)")

In [ ]:
# create yield anomaly - deviation from country-crop average
# this controls for baseline differences between countries

avg_yields = df_merged.groupby(['iso3_code', 'crop'])['yield_kg_ha'].mean().reset_index()
avg_yields.columns = ['iso3_code', 'crop', 'avg_yield']

df_merged = df_merged.merge(avg_yields, on=['iso3_code', 'crop'])
df_merged['yield_anomaly'] = df_merged['yield_kg_ha'] - df_merged['avg_yield']
df_merged['yield_anomaly_pct'] = df_merged['yield_anomaly'] / df_merged['avg_yield'] * 100

print("Yield anomaly stats:")
print(df_merged['yield_anomaly_pct'].describe())

## 4. Research Question 1: Drought Impact on Yields

**How does drought affect crop yields, and which crops are most affected?**

In [ ]:
# compare yields in drought vs non-drought years
drought_impact = df_merged.groupby(['crop', 'drought']).agg({
    'yield_kg_ha': ['mean', 'std', 'count'],
    'yield_anomaly_pct': 'mean'
}).round(2)

drought_impact.columns = ['avg_yield', 'std_yield', 'n_obs', 'avg_anomaly_pct']
drought_impact = drought_impact.reset_index()
drought_impact['drought'] = drought_impact['drought'].map({0: 'Normal', 1: 'Drought'})

print(drought_impact.to_string())

In [ ]:
# calculate yield loss during drought for each crop
# this is kind of messy but it works

crops = df_merged['crop'].unique()
yield_loss = []

for crop in crops:
    crop_data = df_merged[df_merged['crop'] == crop]
    normal = crop_data[crop_data['drought'] == 0]['yield_kg_ha'].mean()
    drought = crop_data[crop_data['drought'] == 1]['yield_kg_ha'].mean()
    loss = (normal - drought) / normal * 100
    
    # t-test to see if difference is significant
    t_stat, p_val = stats.ttest_ind(
        crop_data[crop_data['drought'] == 0]['yield_kg_ha'],
        crop_data[crop_data['drought'] == 1]['yield_kg_ha']
    )
    
    yield_loss.append({
        'crop': crop,
        'normal_yield': normal,
        'drought_yield': drought,
        'yield_loss_pct': loss,
        'p_value': p_val,
        'significant': p_val < 0.05
    })

df_yield_loss = pd.DataFrame(yield_loss)
df_yield_loss = df_yield_loss.sort_values('yield_loss_pct', ascending=False)
print(df_yield_loss.to_string())

In [ ]:
# Visualization: Yield loss by crop
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#d32f2f' if sig else '#90a4ae' for sig in df_yield_loss['significant']]

bars = ax.barh(df_yield_loss['crop'], df_yield_loss['yield_loss_pct'], color=colors)

ax.set_xlabel('Yield Loss During Drought (%)')
ax.set_ylabel('Crop')
ax.set_title('Drought Impact on Crop Yields\n(Red = Statistically Significant, p<0.05)')

# add value labels
for bar, val in zip(bars, df_yield_loss['yield_loss_pct']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, 
            f'{val:.1f}%', va='center', fontsize=10)

ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('../figures/fig1_drought_yield_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved to figures/fig1_drought_yield_loss.png")

In [ ]:
# Box plot comparing drought vs normal years
# Carlos made this one

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
main_crops = ['Wheat', 'Maize', 'Rice', 'Soybeans']

for i, crop in enumerate(main_crops):
    ax = axes[i // 2, i % 2]
    crop_data = df_merged[df_merged['crop'] == crop]
    
    sns.boxplot(data=crop_data, x='drought', y='yield_kg_ha', ax=ax,
                palette=['#4caf50', '#f44336'])
    
    ax.set_xticklabels(['Normal Years', 'Drought Years'])
    ax.set_xlabel('')
    ax.set_ylabel('Yield (kg/ha)')
    ax.set_title(f'{crop}')

plt.suptitle('Yield Distribution: Normal vs Drought Years', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/fig2_drought_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Research Question 2: Regional Vulnerability

**Which regions are most vulnerable to drought?**

In [ ]:
# Calculate regional drought vulnerability
# We'll use yield loss during drought as our main metric

regional_vuln = []

for region in df_merged['region'].dropna().unique():
    region_data = df_merged[df_merged['region'] == region]
    
    # skip if not enough data
    if len(region_data) < 50:
        continue
    
    normal = region_data[region_data['drought'] == 0]['yield_kg_ha'].mean()
    drought = region_data[region_data['drought'] == 1]['yield_kg_ha'].mean()
    
    if normal > 0:  # avoid division by zero
        loss = (normal - drought) / normal * 100
    else:
        loss = 0
    
    # calculate yield variability (CV)
    yield_cv = region_data['yield_kg_ha'].std() / region_data['yield_kg_ha'].mean() * 100
    
    # count drought frequency
    drought_freq = region_data['drought'].mean() * 100
    
    regional_vuln.append({
        'region': region,
        'yield_loss_pct': loss,
        'yield_cv': yield_cv,
        'drought_freq_pct': drought_freq,
        'n_countries': region_data['country_name'].nunique(),
        'n_obs': len(region_data)
    })

df_regional = pd.DataFrame(regional_vuln)
df_regional = df_regional.sort_values('yield_loss_pct', ascending=False)
print(df_regional.to_string())

In [ ]:
# Create vulnerability index
# Combining multiple factors

# normalize each metric to 0-100 scale
def normalize(x):
    return (x - x.min()) / (x.max() - x.min()) * 100

df_regional['yield_loss_norm'] = normalize(df_regional['yield_loss_pct'])
df_regional['yield_cv_norm'] = normalize(df_regional['yield_cv'])
df_regional['drought_freq_norm'] = normalize(df_regional['drought_freq_pct'])

# weighted average - we decided yield loss is most important
df_regional['vulnerability_index'] = (
    0.5 * df_regional['yield_loss_norm'] +
    0.3 * df_regional['yield_cv_norm'] +
    0.2 * df_regional['drought_freq_norm']
)

df_regional = df_regional.sort_values('vulnerability_index', ascending=False)
print("Regional Vulnerability Index:")
print(df_regional[['region', 'vulnerability_index', 'yield_loss_pct', 'yield_cv']].to_string())

In [ ]:
# Visualization: Regional vulnerability
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(df_regional)))

bars = ax.barh(range(len(df_regional)), df_regional['vulnerability_index'], color=colors)
ax.set_yticks(range(len(df_regional)))
ax.set_yticklabels(df_regional['region'])

ax.set_xlabel('Drought Vulnerability Index (0-100)')
ax.set_title('Regional Drought Vulnerability Index')

# add value labels
for i, (bar, val) in enumerate(zip(bars, df_regional['vulnerability_index'])):
    ax.text(val + 1, i, f'{val:.1f}', va='center')

plt.tight_layout()
plt.savefig('../figures/fig3_regional_vulnerability.png', dpi=150)
plt.show()

In [ ]:
# Country-level analysis for top vulnerable region
# Sub-Saharan Africa came out as most vulnerable, let's dig in

top_region = df_regional.iloc[0]['region']
print(f"Analyzing: {top_region}")

country_vuln = []
region_data = df_merged[df_merged['region'] == top_region]

for country in region_data['country_name'].unique():
    country_data = region_data[region_data['country_name'] == country]
    
    if len(country_data) < 20:
        continue
    
    normal = country_data[country_data['drought'] == 0]['yield_kg_ha'].mean()
    drought = country_data[country_data['drought'] == 1]['yield_kg_ha'].mean()
    
    if normal > 0 and not np.isnan(drought):
        loss = (normal - drought) / normal * 100
        country_vuln.append({
            'country': country,
            'yield_loss_pct': loss,
            'avg_yield': country_data['yield_kg_ha'].mean()
        })

df_country_vuln = pd.DataFrame(country_vuln).sort_values('yield_loss_pct', ascending=False)
print(f"\nTop 10 most vulnerable countries in {top_region}:")
print(df_country_vuln.head(10).to_string())

## 6. Research Question 3: Factors of Resilience

**What factors help agricultural systems be more resilient to drought?**

In [ ]:
# Compare high-income vs low-income countries
# Hypothesis: richer countries have more resilience due to irrigation etc

income_impact = []

for income in df_merged['income_group'].dropna().unique():
    inc_data = df_merged[df_merged['income_group'] == income]
    
    normal = inc_data[inc_data['drought'] == 0]['yield_kg_ha'].mean()
    drought = inc_data[inc_data['drought'] == 1]['yield_kg_ha'].mean()
    loss = (normal - drought) / normal * 100 if normal > 0 else 0
    
    income_impact.append({
        'income_group': income,
        'normal_yield': normal,
        'drought_yield': drought,
        'yield_loss_pct': loss,
        'n_obs': len(inc_data)
    })

df_income = pd.DataFrame(income_impact)

# sort by income level
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
df_income['income_group'] = pd.Categorical(df_income['income_group'], categories=income_order, ordered=True)
df_income = df_income.sort_values('income_group')

print(df_income.to_string())

In [ ]:
# visualize income group differences
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left plot: yield loss by income
ax1 = axes[0]
colors = ['#d32f2f', '#f57c00', '#ffc107', '#4caf50']
ax1.bar(range(len(df_income)), df_income['yield_loss_pct'], color=colors)
ax1.set_xticks(range(len(df_income)))
ax1.set_xticklabels(df_income['income_group'], rotation=15, ha='right')
ax1.set_ylabel('Yield Loss During Drought (%)')
ax1.set_title('Drought Impact by Income Group')

# right plot: yield levels
ax2 = axes[1]
x = np.arange(len(df_income))
width = 0.35

ax2.bar(x - width/2, df_income['normal_yield'], width, label='Normal Years', color='#4caf50')
ax2.bar(x + width/2, df_income['drought_yield'], width, label='Drought Years', color='#f44336')

ax2.set_xticks(x)
ax2.set_xticklabels(df_income['income_group'], rotation=15, ha='right')
ax2.set_ylabel('Yield (kg/ha)')
ax2.set_title('Yield Comparison by Income Group')
ax2.legend()

plt.tight_layout()
plt.savefig('../figures/fig4_income_resilience.png', dpi=150)
plt.show()

In [ ]:
# Check if fertilizer use correlates with resilience
# Countries with higher fertilizer use might be more resilient

# need to check if we have fertilizer data
if 'fertilizer_kg_ha' in df_merged.columns:
    # bin countries by fertilizer use
    df_merged['fert_tertile'] = pd.qcut(df_merged['fertilizer_kg_ha'], 3, labels=['Low', 'Medium', 'High'])
    
    fert_impact = []
    for fert in ['Low', 'Medium', 'High']:
        fert_data = df_merged[df_merged['fert_tertile'] == fert]
        normal = fert_data[fert_data['drought'] == 0]['yield_kg_ha'].mean()
        drought = fert_data[fert_data['drought'] == 1]['yield_kg_ha'].mean()
        loss = (normal - drought) / normal * 100
        
        fert_impact.append({
            'fertilizer_use': fert,
            'yield_loss_pct': loss
        })
    
    print(pd.DataFrame(fert_impact))
else:
    print("Fertilizer data not available in merged dataset")

In [ ]:
# Identify resilient countries - low yield loss despite drought exposure
# These are the "success stories"

country_resilience = []

for country in df_merged['country_name'].unique():
    c_data = df_merged[df_merged['country_name'] == country]
    
    if len(c_data) < 30 or c_data['drought'].sum() < 5:  # need enough drought events
        continue
    
    normal = c_data[c_data['drought'] == 0]['yield_kg_ha'].mean()
    drought = c_data[c_data['drought'] == 1]['yield_kg_ha'].mean()
    
    if normal > 0 and not np.isnan(drought):
        loss = (normal - drought) / normal * 100
        
        country_resilience.append({
            'country': country,
            'region': c_data['region'].iloc[0],
            'income_group': c_data['income_group'].iloc[0],
            'yield_loss_pct': loss,
            'drought_events': c_data['drought'].sum(),
            'avg_yield': c_data['yield_kg_ha'].mean()
        })

df_resilience = pd.DataFrame(country_resilience)

# Most resilient (lowest loss)
print("=" * 60)
print("TOP 10 MOST RESILIENT COUNTRIES (lowest yield loss during drought):")
print("=" * 60)
print(df_resilience.nsmallest(10, 'yield_loss_pct').to_string())

print("\n")
print("=" * 60)
print("TOP 10 MOST VULNERABLE COUNTRIES (highest yield loss during drought):")
print("=" * 60)
print(df_resilience.nlargest(10, 'yield_loss_pct').to_string())

## 7. Temporal Trends

Is drought impact getting worse over time?

In [ ]:
# look at trends over decades
df_merged['decade'] = (df_merged['year'] // 10) * 10

decade_trends = []

for decade in sorted(df_merged['decade'].unique()):
    dec_data = df_merged[df_merged['decade'] == decade]
    
    if len(dec_data) < 100:
        continue
    
    normal = dec_data[dec_data['drought'] == 0]['yield_kg_ha'].mean()
    drought = dec_data[dec_data['drought'] == 1]['yield_kg_ha'].mean()
    loss = (normal - drought) / normal * 100 if normal > 0 else 0
    
    decade_trends.append({
        'decade': decade,
        'yield_loss_pct': loss,
        'avg_yield_normal': normal,
        'drought_freq': dec_data['drought'].mean() * 100
    })

df_decades = pd.DataFrame(decade_trends)
print(df_decades.to_string())

In [ ]:
# plot trends
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# left: yield loss trend
ax1 = axes[0]
ax1.plot(df_decades['decade'], df_decades['yield_loss_pct'], marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Decade')
ax1.set_ylabel('Yield Loss During Drought (%)')
ax1.set_title('Drought Impact Over Time')
ax1.grid(True, alpha=0.3)

# right: drought frequency
ax2 = axes[1]
ax2.bar(df_decades['decade'].astype(str), df_decades['drought_freq'], color='#f44336', alpha=0.7)
ax2.set_xlabel('Decade')
ax2.set_ylabel('Drought Frequency (%)')
ax2.set_title('Drought Frequency Over Time')

plt.tight_layout()
plt.savefig('../figures/fig5_temporal_trends.png', dpi=150)
plt.show()

## 8. Summary Statistics and Export

In [ ]:
# create summary table for report
summary = {
    'Total observations': len(df_merged),
    'Countries': df_merged['country_name'].nunique(),
    'Year range': f"{df_merged['year'].min()} - {df_merged['year'].max()}",
    'Crops analyzed': ', '.join(df_merged['crop'].unique()),
    'Drought events (%)': f"{df_merged['drought'].mean()*100:.1f}%",
    'Avg yield (kg/ha)': f"{df_merged['yield_kg_ha'].mean():.0f}",
    'Yield std': f"{df_merged['yield_kg_ha'].std():.0f}"
}

print("Dataset Summary:")
print("-" * 40)
for k, v in summary.items():
    print(f"{k}: {v}")

In [ ]:
# save final dataset
df_merged.to_csv('../data/final_analysis_dataset.csv', index=False)
print(f"Saved final dataset: {len(df_merged)} rows")

# save key result tables
df_yield_loss.to_csv('../data/results_crop_drought_impact.csv', index=False)
df_regional.to_csv('../data/results_regional_vulnerability.csv', index=False)
df_resilience.to_csv('../data/results_country_resilience.csv', index=False)

print("All result tables saved!")

## 9. Key Findings

### Research Question 1: Drought Impact on Crops
- **Maize** shows the highest yield loss during drought (~12%)
- **Rice** is most resilient, likely due to irrigation infrastructure
- All major cereals show statistically significant yield reductions during drought

### Research Question 2: Regional Vulnerability
- **Sub-Saharan Africa** is the most vulnerable region
- Vulnerability driven by both high drought exposure and limited adaptive capacity
- Within SSA, Sahel countries are most at risk

### Research Question 3: Resilience Factors
- **High-income countries** lose only ~5% of yield during drought vs ~15% for low-income
- Investment in irrigation and inputs provides significant buffer
- Some low-income countries show resilience (e.g., Egypt) - likely due to irrigation

### Limitations
- Drought defined using precipitation only (not considering temperature stress)
- Country-level analysis masks local variation
- Missing data for some countries/years
- Cannot establish causation from observational data

### Policy Recommendations
1. Prioritize investment in drought-resistant maize varieties
2. Expand irrigation infrastructure in Sub-Saharan Africa
3. Develop early warning systems for drought-prone regions
4. Support agricultural insurance programs in vulnerable areas

In [ ]:
print("Analysis complete!")
print(f"\nFiles created:")
print("- figures/fig1_drought_yield_loss.png")
print("- figures/fig2_drought_boxplots.png")
print("- figures/fig3_regional_vulnerability.png")
print("- figures/fig4_income_resilience.png")
print("- figures/fig5_temporal_trends.png")
print("- data/final_analysis_dataset.csv")
print("- data/results_*.csv")